Imports and Setup

In [ ]:
# import libs
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

# variables
BATCH_SIZE = 64
TEST_BATCH_SIZE = 10
LEARNING_RATE = 0.001
EPOCHS = 20
SEED = 42

# set seed
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)

# find GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# print it
print(f'Device: {device}')

Data Uploading

In [ ]:
# data function
def get_dataloaders(batch_size):
    # format tensors
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    # download images
    train_full = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
    test_dataset = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

    train_size = int(0.85 * len(train_full))
    val_size = len(train_full) - train_size

    # split data
    train_dataset, val_dataset = random_split(train_full, [train_size, val_size])

    # loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_dataset,  batch_size=TEST_BATCH_SIZE, shuffle=False)
    return train_loader, val_loader, test_loader

Model Architecture

In [ ]:
class SpiderNet(nn.Module):
    # init layers
    def __init__(self):
        # super init
        super().__init__()
        # flatten input
        self.flatten = nn.Flatten()
        # first layer
        self.fc1 = nn.Linear(784, 16)

        # left branch
        self.left1 = nn.Linear(16, 8)
        self.left2 = nn.Linear(8, 8)

        # right branch
        self.right1 = nn.Linear(16, 12)
        self.right2 = nn.Linear(12, 8)

        # final output
        self.output = nn.Linear(16, 10)
        # remove negative ones
        self.relu = nn.ReLU()

    # Forward pass
    def forward(self, x):
        # flatten input
        x = self.flatten(x)
        # first pass
        x = self.relu(self.fc1(x))

        # left path
        l1_out = self.relu(self.left1(x))
        l2_out = self.relu(self.left2(l1_out))
        # skip connection
        left_final = l1_out + l2_out

        # Right path
        r1_out = self.relu(self.right1(x))
        right_final = self.relu(self.right2(r1_out))

        # merge branches
        merged = torch.cat([left_final, right_final], dim=1)
        # output
        out = self.output(merged)
        return out

Training Engine

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    # train mode
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    # loop batches
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        # clear grads
        optimizer.zero_grad()
        # get preds
        preds = model(images)
        # calc loss
        loss = criterion(preds, labels)
        # back propagation
        loss.backward()
        # update weights
        optimizer.step()

        # add loss
        running_loss += loss.item() * images.size(0)
        # get guesses
        predicted = torch.argmax(preds, dim=1)
        # count total
        total += labels.size(0)
        # count correct ones
        correct += (predicted == labels).sum().item()

    return running_loss / total, 100. * correct / total

Evaluate

In [ ]:
def evaluate_model(model, dataloader, criterion, device):
    # eval mode
    model.eval()
    running_loss, correct, total = 0.0, 0, 0

    # stop grads
    with torch.no_grad():
        # loop batches
        for images, labels in dataloader:
            # move data
            images, labels = images.to(device), labels.to(device)
            # get preds
            preds = model(images)
            # calc loss
            loss = criterion(preds, labels)

            running_loss += loss.item() * images.size(0)
            # get guesses
            predicted = torch.argmax(preds, dim=1)
            # count total
            total += labels.size(0)
            # count correct ones
            correct += (predicted == labels).sum().item()

    return running_loss / total, 100. * correct / total

Plotting

In [ ]:
def plot_metrics(train_losses, val_losses, train_accs, val_accs):
    # make figure
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # plot loss
    ax1.plot(train_losses, label='Train Loss', color='blue')
    ax1.plot(val_losses, label='Val Loss', color='orange')
    # title
    ax1.set_title('Loss')
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Loss')
    # show legend
    ax1.legend()

    # plot acc
    ax2.plot(train_accs, label='Train Acc', color='green')
    ax2.plot(val_accs, label='Val Acc', color='red')
    # title
    ax2.set_title('Accuracy')
    ax2.set_xlabel('Epochs')
    ax2.set_ylabel('Accuracy (%)')
    # show legend
    ax2.legend()

    # fix layout
    plt.tight_layout()
    # save img
    plt.savefig('metrics_plot.png', dpi=150)
    plt.show()

Error Analysis

In [ ]:
def plot_misclassifications(model, dataloader, device):
    # class names
    names = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
             'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Boot']

    # eval mode
    model.eval()
    # empty lists
    misclassified_imgs, true_labels, pred_labels = [], [], []

    # stop grads
    with torch.no_grad():
        # loop batches
        for images, labels in dataloader:
            # data
            images, labels = images.to(device), labels.to(device)
            # preds
            preds = model(images)
            # guesses
            predicted = torch.argmax(preds, dim=1)

            # find errors
            errors = (predicted != labels)

            if errors.any():
                # save
                misclassified_imgs.append(images[errors].cpu())
                true_labels.extend(labels[errors].cpu().numpy())
                pred_labels.extend(predicted[errors].cpu().numpy())

            # check limit
            if len(true_labels) >= 10:
                break

    # merge tensors
    misclassified_imgs = torch.cat(misclassified_imgs)
    # make grid
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    # flatten axes
    axes = axes.flatten()

    # loop 10
    for i in range(10):
        # drop dim
        img = np.squeeze(misclassified_imgs[i].numpy())
        # show img
        axes[i].imshow(img, cmap='gray')

        # get names
        true_name = names[true_labels[i]]
        pred_name = names[pred_labels[i]]
        # set titles
        axes[i].set_title(f"T: {true_name}\nP: {pred_name}", color='red')
        # hide axes
        axes[i].axis('off')

    # fix layout
    plt.tight_layout()
    # save img
    plt.savefig('misclassifications.png', dpi=150)
    plt.show()

Confusion matrix

In [ ]:
def plot_confusion_matrix(model, dataloader, device):
    # class names
    names = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
             'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Boot']
    
    # eval mode
    model.eval()
    
    # initialize tensor
    conf_matrix = torch.zeros(10, 10, dtype=torch.int64, device=device)
    
    # disable gradients
    with torch.no_grad():
        # loop batches
        for images, labels in dataloader:
            # move data
            images, labels = images.to(device), labels.to(device)
            
            # predictions
            predicted = torch.argmax(model(images), dim=1)
            
            # update matrix
            conf_matrix[labels, predicted] += 1

    # to CPU
    cm = conf_matrix.cpu().numpy()

    fig, ax = plt.subplots(figsize=(10, 10))
    
    # display matrix
    im = ax.imshow(cm, cmap='Blues')
    
    # add colorbar
    plt.colorbar(im, ax=ax)

    ax.set_xticks(range(10))
    ax.set_yticks(range(10))
    
    # label x-axis
    ax.set_xticklabels(names, rotation=45, ha='right')
    
    # label y-axis
    ax.set_yticklabels(names)
    
    # name x-axis
    ax.set_xlabel('Predicted')
    
    # name y-axis
    ax.set_ylabel('True')
    
    # add title
    ax.set_title('Confusion Matrix')

    # row loop
    for i in range(10):
        # column loop
        for j in range(10):
            # insert text
            ax.text(j, i, str(cm[i][j]),
                    ha='center', va='center',
                    color='white' if cm[i][j] > cm.max() / 2 else 'black')

    # optimize layout
    plt.tight_layout()
    
    # save image
    plt.savefig('confusion_matrix.png', dpi=150)
    
    plt.show()

Submissions

In [ ]:
if __name__ == "__main__":
    # get data
    train_loader, val_loader, test_loader = get_dataloaders(BATCH_SIZE)

    # init model
    model = SpiderNet().to(device)
    # loss
    criterion = nn.CrossEntropyLoss()
    # optim
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    # empty lists
    t_loss, v_loss_list, t_acc, v_acc = [], [], [], []
    # best
    best_val_acc = 0.0

    # Loop epochs
    for epoch in range(EPOCHS):
        # train model
        train_l, train_a = train_epoch(model, train_loader, criterion, optimizer, device)
        # eval model
        val_l, val_a = evaluate_model(model, val_loader, criterion, device)

        t_loss.append(train_l)
        t_acc.append(train_a)
        v_loss_list.append(val_l)
        v_acc.append(val_a)

        # print stats
        print(f"E {epoch+1}/{EPOCHS} | TL: {train_l:.4f} TA: {train_a:.2f}% | VL: {val_l:.4f} VA: {val_a:.2f}%")

        # check best
        if val_a > best_val_acc:
            best_val_acc = val_a
            torch.save(model.state_dict(), 'best_model_weights.pth')
            print(f"Saved! (Acc: {best_val_acc:.2f}%)")

    print("\n Done")

    # plot stats
    plot_metrics(t_loss, v_loss_list, t_acc, v_acc)

    model.load_state_dict(torch.load('best_model_weights.pth', weights_only=True))
    print("Loaded best.")

    # eval mode
    model.eval()
    # empty list
    test_preds = []

    # stop grads
    with torch.no_grad():
        for images, _ in test_loader:
            # data
            images = images.to(device)
            # preds
            preds = model(images)
            # guesses
            predicted = torch.argmax(preds, dim=1)
            # save
            test_preds.extend(predicted.cpu().numpy())

    # make CSV
    pd.DataFrame({
        'Id': range(len(test_preds)),
        'Category': test_preds
    }).to_csv('submission.csv', index=False)
    # print CSV
    print("Saved CSV.")

    # plot
    print("Plotting errors...")
    plot_misclassifications(model, val_loader, device)